# React Agent - Google ADK

## Load Keys from Env.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Tools

In [3]:
from datetime import date, datetime, timedelta


def calculator(expression: str) -> dict:
    """Evaluates a Python expression and returns the result.

    Supports arithmetic (e.g. '23 * 47') AND date math via date.fromisoformat('YYYY-MM-DD').
    To compute days between two dates:
        (date.fromisoformat('2026-09-14') - date.fromisoformat('2026-05-14')).days

    Do NOT pass raw quoted date strings like "'2026-09-14' - '2026-05-14'" —
    that's string subtraction and will fail.

    ONLY pass date expressions or arithmetic expressions.

    Args:
        expression: A Python expression to evaluate.
    """
    # Expose date/datetime/timedelta so the model can do date math too.
    # NEVER use eval() on untrusted input in real code — for a demo it's fine.
    safe_globals = {
        "__builtins__": {},
        "date":      date,
        "datetime":  datetime,
        "timedelta": timedelta,
        "abs": abs, "min": min, "max": max, "round": round,
    }
    return {"result": eval(expression, safe_globals, {})}


KB = {
    "price_per_ticket": 89,
    "attendees":        47,
    "conference_date":  "2026-09-14",
    "venue":            "Lisbon Congress Center",
}

def lookup(key: str) -> dict:
    """Looks up a fact in the conference knowledge base.

    Args:
        key: The key to look up. Available keys: 'attendees', 'price_per_ticket', 'conference_date', 'venue'.
    """
    return {
        "value": KB.get(key, f"unknown key: {key}"),
        "available_keys": list(KB.keys()),
    }


def get_today() -> dict:
    """Returns today's date in ISO format (YYYY-MM-DD).

    Use this whenever the question depends on knowing the current date — relative
    dates, days until an event, etc.
    """
    return {"today": date.today().isoformat()}

## Agent

In [4]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

agent = LlmAgent(
    name="conference_assistant",
    model=LiteLlm(model="cohere_chat/command-a-03-2025"),
    instruction=(
        "You are a precise assistant. Use the tools to gather facts and to do any "
        "arithmetic. When you have enough information, stop calling tools and answer."
        "When calling the calculator tool, ALWAYS pass a single-line expression with no "
        "comments, no newlines, and no variable assignments."
    ),
    tools=[
        calculator,
        lookup,
        get_today,
    ],
)

print("Agent Ready ✅")

Agent Ready ✅


## Runner

In [13]:
from google.adk.runners import InMemoryRunner
from google.genai import types

APP_NAME = "grokking_demo"
USER_ID = "student"


async def ask(agent, question: str) -> str:
    runner = InMemoryRunner(agent=agent, app_name=APP_NAME)
    session = await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID
    )

    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_text = ""

    async for event in runner.run_async(
        user_id=USER_ID, session_id=session.id, new_message=content
    ):
        # Print the trace: tool calls + tool results, like Lab 5 did.
        if event.content and event.content.parts:
            for p in event.content.parts:
                if getattr(p, "function_call", None):
                    print(f"  → {p.function_call.name}({dict(p.function_call.args)})")
                if getattr(p, "function_response", None):
                    print(f"        = {p.function_response.response}")
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text or ""

    return final_text

## User Input

In [14]:
question = (
    "What's the total ticket revenue for the conference, when and where is it happening, "
    "and how many days from today until it starts?"
)

answer = await ask(agent, question)
print("\n=== FINAL ANSWER ===\n", answer)

/Users/ABhadoria/Work/ai-stuff/grokking-agents/grokking-agents/.venv/lib/python3.14/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


  → lookup({'key': 'attendees'})
  → lookup({'key': 'price_per_ticket'})
  → lookup({'key': 'conference_date'})
  → lookup({'key': 'venue'})
  → get_today({})
        = {'value': 47, 'available_keys': ['price_per_ticket', 'attendees', 'conference_date', 'venue']}
        = {'value': 89, 'available_keys': ['price_per_ticket', 'attendees', 'conference_date', 'venue']}
        = {'value': '2026-09-14', 'available_keys': ['price_per_ticket', 'attendees', 'conference_date', 'venue']}
        = {'value': 'Lisbon Congress Center', 'available_keys': ['price_per_ticket', 'attendees', 'conference_date', 'venue']}
        = {'today': '2026-06-07'}
  → calculator({'expression': '47*89'})
  → calculator({'expression': "(date.fromisoformat('2026-09-14') - date.fromisoformat('2026-06-07')).days"})
        = {'result': 4183}
        = {'result': 99}

=== FINAL ANSWER ===
 The total ticket revenue for the conference is 4183, the conference is happening on 2026-09-14 at the Lisbon Congress Center, and i